In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "sshleifer/distilbart-cnn-12-6"
summ_tokenizer = AutoTokenizer.from_pretrained(model_name)
summ_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [2]:
article = """
The Reserve Bank of India kept its policy repo rate unchanged for the third consecutive
meeting, citing persistent inflationary pressures from food prices even as core inflation
continues to ease. The central bank's monetary policy committee voted unanimously to
maintain the current stance, signaling that any rate cuts remain some way off. Governor
emphasized that while growth indicators remain resilient, the committee wants to see
sustained evidence that inflation is durably aligned with the target before considering
any easing. Market participants had largely priced in this decision, though some economists
had expected a more dovish tone given recent softening in industrial output data.
"""

inputs = summ_tokenizer(article, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

with torch.no_grad():
    summary_ids = summ_model.generate(
        **inputs,
        max_length=60,
        min_length=20,
        num_beams=4
    )

summary = summ_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

 The Reserve Bank of India kept its policy repo rate unchanged for the third consecutive meeting . The central bank's monetary policy committee voted unanimously to maintain the current stance, signaling that any rate cuts remain some way off .


In [4]:
from transformers import AutoTokenizer as QwenTokenizer, AutoModelForCausalLM

qwen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = QwenTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_name, torch_dtype=torch.float16).to("cuda")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [6]:
messages = [
    {"role": "user", "content": f"Summarize this in 2 sentences:\n\n{article}"}
]
qwen_prompt = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
qwen_inputs = qwen_tokenizer(qwen_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    qwen_output = qwen_model.generate(**qwen_inputs, max_new_tokens=80, do_sample=False)

qwen_summary = qwen_tokenizer.decode(qwen_output[0][qwen_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(qwen_summary)

The Reserve Bank of India maintained its policy interest rate at unchanged levels for the third time consecutively due to persistently high inflation driven by food costs despite declining core inflation rates. The central bank's Monetary Policy Committee decided against further rate adjustments, indicating they will wait until there is sustained evidence of durable inflation alignment with their target before making any changes.


In [8]:
from transformers import MarianMTModel, MarianTokenizer

trans_model_name = "Helsinki-NLP/opus-mt-en-hi"
trans_tokenizer = MarianTokenizer.from_pretrained(trans_model_name)
trans_model = MarianMTModel.from_pretrained(trans_model_name).to("cuda")

text_to_translate = "Understanding transformer models is essential for building modern AI applications."
inputs = trans_tokenizer(text_to_translate, return_tensors="pt").to("cuda")

with torch.no_grad():
    translated_ids = trans_model.generate(**inputs, num_beams=4, max_length=60)

translated_text = trans_tokenizer.decode(translated_ids[0], skip_special_tokens=True)
print(translated_text)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

आधुनिक एआई अनुप्रयोगों को बनाने के लिए रूपांतरण मॉडलों को समझना ज़रूरी है ।
